In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

required_settings = {
    "GROQ_API_KEY": GROQ_API_KEY,
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_USERNAME": NEO4J_USERNAME,
    "NEO4J_PASSWORD": NEO4J_PASSWORD,
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise RuntimeError(f"Missing settings in .env: {', '.join(missing_settings)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.7/390.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.5/301.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.14.2
    Uninstalling langsmith-0.14.2:
      Successfully uninstalled langsmith-0.14.2
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.6
    Uninstalling langc

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [16]:
from langchain_core.prompts import PromptTemplate

qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You answer questions using data retrieved from a graph database.

The user's question is:
{question}

The database returned these records:
{context}

Instructions:
- Read the returned records carefully.
- The records contain the answer if they include relevant values.
- For movie titles, return the movie title or titles.
- Never say "I don't know" when the records contain the answer.
- Only say "I don't know" if the records contain no relevant information.

Answer:
"""
)


cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template="""
You generate Cypher queries for a Neo4j knowledge graph.

Graph schema:
{schema}

Rules:
- Use only labels, relationships, and properties present in the schema.
- Match entity and genre names to their actual stored values.
- Treat equivalent wording, hyphenation, and spacing as natural-language variations.
  For example, "science-fiction" and "science fiction" may refer to
  the same genre if the graph stores "Science Fiction".
- Follow the relationships in the schema to answer the complete question.
- Return only a valid, read-only Cypher query.
- Do not create, update, or delete graph data.

Question: {question}

Cypher query:
"""
)

In [ ]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain


# Connect to Neo4j
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD
)

/usr/local/lib/python3.13/dist-packages/langchain_neo4j/graphs/neo4j_graph.py:158: PreviewWarning: The Query API/HTTP support in the Neo4j Python driver is currently in preview. It might be changed without following the deprecation policy. See also https://github.com/neo4j/neo4j-python-driver/wiki/preview-features.
  self._driver = neo4j.GraphDatabase.driver(


In [21]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=cypher_prompt,  # your existing prompt, if defined
    qa_prompt=qa_prompt,
    verbose=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=True,
)


In [31]:
question = input("Ask a question about movies: ")

result = chain.invoke({"query": question})

print(result["result"])

Ask a question about movies: I like movie Dunkirk, Can you recommend some other movies?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Dunkirk'})
OPTIONAL MATCH (m)<-[:ACTED_IN]-(p:Person)-[:ACTED_IN]->(rec:Movie)
OPTIONAL MATCH (m)<-[:DIRECTED]-(d:Person)-[:DIRECTED]->(rec)
OPTIONAL MATCH (m)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec)
WHERE rec.title <> 'Dunkirk'
RETURN DISTINCT rec.title AS recommendation
ORDER BY rec.title
LIMIT 10
Full Context:
[{'recommendation': 'Inception'}, {'recommendation': 'The Dark Knight'}]

> Finished chain.
Here are a couple of movies you might enjoy:

- **Inception**  
- **The Dark Knight**
